In [ ]:
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as mpatches
from ultralytics import YOLO

# paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "models" else Path.cwd()
CHECKPOINTS = PROJECT_ROOT / "models" / "checkpoints"
YAMLS = PROJECT_ROOT / "data" / "processed" / "yamls"
STAGE3_YAML = YAMLS / "stage3_disease.yaml"
TEST_IMGS = PROJECT_ROOT / "data" / "processed" / "stage3_disease" / "images" / "test"
RESULTS_DIR = PROJECT_ROOT / "results" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {0: "Impacted", 1: "Caries", 2: "Periapical Lesion", 3: "Deep Caries"}
CLASS_COLORS = {0: "#e07b54", 1: "#5b8dd9", 2: "#59a96a", 3: "#c0392b"}

def get_device():
    if torch.backends.mps.is_available(): return "mps"
    if torch.cuda.is_available(): return "cuda"
    return "cpu"

DEVICE = get_device()
print(f"Device: {DEVICE}")
print(f"Project root: {PROJECT_ROOT}")

Device       : mps
Project root : /Users/brycegrover/Desktop/DSAN/SPRING_2026/NeuralNets-6600/project/Dental_Project


In [ ]:
# check checkpoints
ckpts = {
    "Curriculum (Stage 3)": CHECKPOINTS / "stage3_best.pt",
    "Baseline": CHECKPOINTS / "baseline_best.pt",
    "Stage 1": CHECKPOINTS / "stage1_best.pt",
    "Stage 2": CHECKPOINTS / "stage2_best.pt",
}
for name, path in ckpts.items():
    status = "" if path.exists() else "MISSING — run training first"
    print(f" {status}, {name}: {path.name}")

  ✓  Curriculum (Stage 3): stage3_best.pt
  ✓  Baseline: baseline_best.pt
  ✓  Stage 1: stage1_best.pt
  ✓  Stage 2: stage2_best.pt


In [ ]:
# get the metrics for each model
def get_metrics(ckpt_path, yaml_path, split="val", name="model"):
    if not ckpt_path.exists():
        print(f"  Skipping {name}: checkpoint not found.")
        return None
    model   = YOLO(str(ckpt_path))
    metrics = model.val(
        data=str(yaml_path),
        split=split,
        device=DEVICE,
        verbose=False,
    )
    return metrics

In [ ]:
# Validate each stage on the stage3 val set
val_metrics = {}
for name, path in ckpts.items():
    print(f"Validating {name}...")
    m = get_metrics(path, STAGE3_YAML, split="val", name=name)
    if m is not None:
        val_metrics[name] = m

Validating Curriculum (Stage 3)...
Ultralytics 8.4.18 🚀 Python-3.12.9 torch-2.5.1 MPS (Apple M3 Max)
YOLOv8m-seg summary (fused): 106 layers, 27,224,700 parameters, 0 gradients, 110.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5929.1±3197.2 MB/s, size: 3336.6 KB)
val: Scanning /Users/brycegrover/Desktop/DSAN/SPRING_2026/NeuralNets-6600/project/Dental_Project/data/processed/stage3_disease/labels/val... 50 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 50/50 125.3it/s 0.4s.2s
val: New cache created: /Users/brycegrover/Desktop/DSAN/SPRING_2026/NeuralNets-6600/project/Dental_Project/data/processed/stage3_disease/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.5s/it 18.1s7.6s2s
                   all         50        182      0.527      0.601      0.543      0.363      0.494      0.559      0.518      0.359
Speed: 1.7ms preprocess, 58.5ms inferen

In [ ]:
# val summary table
rows = []
for name, m in val_metrics.items():
    rows.append({
        "Model": name,
        "mAP@0.5 (box)": round(m.box.map50,  4),
        "mAP@0.5:0.95": round(m.box.map, 4),
        "mAP@0.5 (seg)": round(m.seg.map50, 4) if hasattr(m, "seg") else "—",
    })

df_val = pd.DataFrame(rows)
print("Validation set metrics")
print(df_val.to_string(index=False))


── Validation set metrics ──
               Model  mAP@0.5 (box)  mAP@0.5:0.95  mAP@0.5 (seg)
Curriculum (Stage 3)         0.5425        0.3633         0.5182
            Baseline         0.6049        0.4064         0.5857
             Stage 1         0.0288        0.0156         0.0288
             Stage 2         0.0260        0.0151         0.0260


In [ ]:
test_models = {
    "Curriculum (Stage 3)": CHECKPOINTS / "stage3_best.pt",
    "Baseline": CHECKPOINTS / "baseline_best.pt",
}
test_metrics = {}
for name, path in test_models.items():
    print(f"Testing {name}...")
    m = get_metrics(path, STAGE3_YAML, split="test", name=name)
    if m is not None:
        test_metrics[name] = m

Testing Curriculum (Stage 3)...
Ultralytics 8.4.18 🚀 Python-3.12.9 torch-2.5.1 MPS (Apple M3 Max)
YOLOv8m-seg summary (fused): 106 layers, 27,224,700 parameters, 0 gradients, 110.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5372.1±1897.6 MB/s, size: 3452.1 KB)
val: Scanning /Users/brycegrover/Desktop/DSAN/SPRING_2026/NeuralNets-6600/project/Dental_Project/data/processed/stage3_disease/labels/test... 250 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 250/250 106.6it/s 2.3s0.0s
val: New cache created: /Users/brycegrover/Desktop/DSAN/SPRING_2026/NeuralNets-6600/project/Dental_Project/data/processed/stage3_disease/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 4.1s/it 1:054.2sss
                   all        250       1469      0.401      0.441      0.394      0.273      0.392      0.435      0.386      0.257
Speed: 0.7ms preprocess, 10.3ms inf

In [ ]:
# RQ1 comparison table
rows = []
for name, m in test_metrics.items():
    rows.append({
        "Model": name,
        "mAP@0.5 (box)": round(m.box.map50, 4),
        "mAP@0.5:0.95 (box)": round(m.box.map, 4),
        "mAP@0.5 (seg)": round(m.seg.map50, 4) if hasattr(m, "seg") else "—",
        "mAP@0.5:0.95 (seg)": round(m.seg.map, 4) if hasattr(m, "seg") else "—",
    })

df_test = pd.DataFrame(rows)
print("Test set metrics (RQ1)")
print(df_test.to_string(index=False))


── Test set metrics (RQ1) ──
               Model  mAP@0.5 (box)  mAP@0.5:0.95 (box)  mAP@0.5 (seg)  mAP@0.5:0.95 (seg)
Curriculum (Stage 3)         0.3944              0.2729         0.3857              0.2570
            Baseline         0.4166              0.2830         0.4002              0.2588


In [8]:
# RQ1 bar chart
metrics_to_plot = ["mAP@0.5 (box)", "mAP@0.5:0.95 (box)", "mAP@0.5 (seg)"]
x   = np.arange(len(metrics_to_plot))
w   = 0.3
colors = ["#2c7bb6", "#d7191c"]

fig, ax = plt.subplots(figsize=(9, 5))
for i, (name, row) in enumerate(zip(test_metrics.keys(), rows)):
    vals = [row.get(m, 0) for m in metrics_to_plot]
    bars = ax.bar(x + i * w, vals, w, label=name, color=colors[i], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + w / 2)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylabel("mAP")
ax.set_title("RQ1 — Curriculum vs. Baseline (Test Set)", fontweight="bold")
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "rq1_curriculum_vs_baseline.png", dpi=150)
plt.show()

<Figure size 900x500 with 1 Axes>

In [ ]:
# to get per-class AP tables, we can extract the relevant info from the metrics object and put it in a DataFrame
def per_class_df(metrics, class_names):
    if metrics is None:
        return None
    ap50 = metrics.box.ap50 # shape: (nc,)
    ap50_95 = metrics.box.ap # shape: (nc,)
    classes = list(range(len(ap50)))
    return pd.DataFrame({
        "Class": [class_names.get(c, str(c)) for c in classes],
        "AP@0.5": [round(v, 4) for v in ap50],
        "AP@0.5:0.95": [round(v, 4) for v in ap50_95],
    })

if "Curriculum (Stage 3)" in test_metrics:
    df_cls_curr = per_class_df(test_metrics["Curriculum (Stage 3)"], CLASS_NAMES)
    df_cls_base = per_class_df(test_metrics.get("Baseline"), CLASS_NAMES)

── Curriculum per-class AP@0.5 ──
            Class  AP@0.5  AP@0.5:0.95
         Impacted  0.9424       0.6326
           Caries  0.4212       0.3136
Periapical Lesion  0.1928       0.1293
      Deep Caries  0.0213       0.0161


In [ ]:
# per-class grouped bar chart
if df_cls_curr is not None and df_cls_base is not None:
    classes = df_cls_curr["Class"].tolist()
    x  = np.arange(len(classes))
    w  = 0.3

    fig, ax = plt.subplots(figsize=(10, 5))
    b1 = ax.bar(x - w/2, df_cls_curr["AP@0.5"], w, label="Curriculum", color="#2c7bb6", alpha=0.85)
    b2 = ax.bar(x + w/2, df_cls_base["AP@0.5"], w, label="Baseline", color="#d7191c", alpha=0.85)

    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.008,
                f"{h:.3f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(classes)
    ax.set_ylabel("AP@0.5")
    ax.set_title("RQ3 — Per-Class Performance (Test Set)", fontweight="bold")
    ax.legend()
    ax.set_ylim(0, 1.0)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "rq3_per_class_ap.png", dpi=150)
    plt.show()

<Figure size 1000x500 with 1 Axes>

In [11]:
# Extract val mAP@0.5 per stage (box)
stage_order = ["Baseline", "Stage 1", "Stage 2", "Curriculum (Stage 3)"]
stage_map50 = []
for label in stage_order:
    m = val_metrics.get(label)
    if m is not None:
        stage_map50.append((label, m.box.map50))
    else:
        stage_map50.append((label, None))

labels, vals = zip(*[(l, v) for l, v in stage_map50 if v is not None])

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, vals, color=["#d7191c", "#fdae61", "#abd9e9", "#2c7bb6"], alpha=0.85, width=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.008,
            f"{v:.3f}", ha="center", va="bottom", fontsize=10)

ax.set_ylabel("mAP@0.5 (box, val set)")
ax.set_title("RQ2 — Stage-by-stage Contribution (Val Set)", fontweight="bold")
ax.set_ylim(0, max(vals) * 1.2)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "rq2_stage_contribution.png", dpi=150)
plt.show()

<Figure size 800x400 with 1 Axes>

## 5. Visual predictions on sample X-rays

In [ ]:
# Run inference on a sample of test images
curriculum_ckpt = CHECKPOINTS / "stage3_best.pt"
if not curriculum_ckpt.exists():
    print("Curriculum checkpoint not found.")
else:
    sample_imgs = sorted(TEST_IMGS.glob("*.png"))[:6]
    model = YOLO(str(curriculum_ckpt))
    preds = model.predict(
        source=[str(p) for p in sample_imgs],
        device=DEVICE,
        conf=0.25,
        iou=0.45,
        verbose=False,
    )

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for ax, result in zip(axes, preds):
        img_bgr = result.plot(conf=True, line_width=2)
        img_rgb = img_bgr[:, :, ::-1]   # BGR RGB
        ax.imshow(img_rgb)
        ax.set_title(Path(result.path).stem, fontsize=9)
        ax.axis("off")

    # Legend
    patches = [mpatches.Patch(color=v, label=k) for k, v in CLASS_COLORS.items()]
    fig.legend(handles=patches, loc="lower center", ncol=4, fontsize=10,
               title="Diagnosis", frameon=True)
    plt.suptitle("Curriculum Model — Sample Predictions", fontsize=13, fontweight="bold")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.savefig(RESULTS_DIR / "sample_predictions.png", dpi=150)
    plt.show()

<Figure size 1500x800 with 6 Axes>

In [ ]:
# conf matrices
run_dirs = {
    "Curriculum": CHECKPOINTS / "runs" / "stage3_curriculum",
    "Baseline": CHECKPOINTS / "runs" / "baseline",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (label, run_dir) in zip(axes, run_dirs.items()):
    cm_path = run_dir / "confusion_matrix_normalized.png"
    if not cm_path.exists():
        cm_path = run_dir / "confusion_matrix.png"
    if cm_path.exists():
        img = mpimg.imread(cm_path)
        ax.imshow(img)
        ax.set_title(f"Confusion Matrix — {label}", fontsize=11)
    else:
        ax.text(0.5, 0.5, f"Not yet available\n{run_dir.name}",
                ha="center", va="center", transform=ax.transAxes, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrices.png", dpi=150)
plt.show()

<Figure size 1400x600 with 2 Axes>